# Timing Analysis

## What you'll learn

- Load pipeline timing data with `PipelineTimings.from_delta()`
- Inspect step-level phase timings as a DataFrame
- Plot stacked bar charts of step phase breakdowns
- Drill into per-execution timings for individual steps
- Compare mean execution timings across steps
- Inspect subprocess launch durations and missing command evidence

Every pipeline step records timing information for each execution phase:
resolving inputs, batching and cache checking, executing operations,
committing results, and compacting tables. **PipelineTimings** loads this
data from the delta store and provides DataFrames and plots for
identifying bottlenecks.

**Prerequisites:** [Your First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb).  
**Estimated time:** 10 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager, Runner
from artisan.utils import tutorial_setup
from artisan.visualization import PipelineTimings

In [ ]:
env = tutorial_setup("timing_analysis")

Build a multi-step pipeline to generate timing data. We use batching on
the DataTransformer step to create multiple execution units, which makes
the per-execution analysis more interesting.

In [ ]:
pipeline = PipelineManager.create(
    name="timing_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 10, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 5},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("transform", "dataset")},
    params={
        "criteria": [
            {"metric": "distribution.median", "operator": "gt", "value": 0.3},
        ]
    },
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="refine",
    inputs={"dataset": output("filter", "passthrough")},
    params={"seed": 99},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()

## Load timing data

Load timing data from the delta store with `PipelineTimings.from_delta()`.
This reads the steps and executions tables, parses the embedded timing
metadata, and structures it for analysis. For a run-scoped query, cached
source durations are excluded because that work happened in an earlier run.
Execution timing includes fresh successful executions from succeeded or partial
steps. Use `inspect_commands` for command evidence from failed attempts.

In [ ]:
timings = PipelineTimings.from_delta(env.delta_root)

## Step-level timings

The `.step_timings()` method returns a DataFrame with one row per step.
Each row includes the step number, name, total duration, and one column
per timing phase. The phases vary by operation type — creator steps
include phases like `capture_logs`, while curator steps do not.

In [ ]:
timings.step_timings()

## Plot step phase breakdown

The `.plot_steps()` method renders a stacked horizontal bar chart showing
how each step's time breaks down by phase. This is the fastest way to
spot which steps are slow and why.

In [ ]:
timings.plot_steps()

You can filter to specific steps with the `step_numbers` parameter --
useful for focusing on a subset of a long pipeline.

In [ ]:
timings.plot_steps(step_numbers=[0, 1, 2])

## Execution-level timings

Step-level timings tell you *which* step is slow. **Execution-level
timings** tell you *why* — they break down each individual execution
unit within a step. An execution unit is one batch of artifacts processed
together — when a step uses batching, it produces multiple execution
units that can run in parallel.

Note that execution-level phases differ from step-level phases. Step
phases track the orchestration lifecycle (resolve inputs, batch, commit,
etc.), while execution phases track the operation lifecycle (setup,
execute, record, etc.).

In [ ]:
timings.execution_timings(step_number=1)

Each row is one execution unit. Step 1 used `artifacts_per_unit=5` with
10 inputs, so you should see 2 execution units.

## Subprocess launch timings

Inspect subprocess launch evidence separately from execution phases:


In [ ]:
timings.command_timings(step_number=1)

These Python examples launch no subprocesses, so they have a complete empty
recording and no `launch_seconds`. A command operation records time spent
creating its subprocess. That duration does not measure tool readiness, queueing,
or time waiting for output, and it is already included in the execution phases.

A missing duration can also mean evidence was unavailable or a launch failed.
Check `recording_status` before treating a null value as “no subprocess.” Use
`inspect_commands` for the recorded command details; see
[Inspect Pipeline Results](../../how-to-guides/inspecting-provenance.md).

## Execution statistics

For steps with many execution units, summary statistics are more useful
than raw timings. The `.execution_stats()` method computes mean, standard
deviation, min, and max for each phase across all execution units in a
step. Step 1 has two units: enough to illustrate the calculation, but too few
to draw broad performance conclusions.

In [ ]:
timings.execution_stats(step_number=1)

## Compare execution timings across steps

The `.plot_execution_stats()` method compares mean execution timings
across all steps. This helps answer: *Which operation is the slowest
per execution unit?*

In [ ]:
timings.plot_execution_stats()

## Summary

Start with step timings to locate expensive work, then inspect the units within
that step. Phase timings distinguish preparation, execution, and persistence
costs. Command launch timings answer a narrower question and should not be
added to phase totals. Measure representative workloads before changing batching
or concurrency.

## Next steps

- [Batching and Performance](../04-batching/01-batching-and-performance.ipynb) -- Tune batching parameters based on timing insights
- [Provenance Graphs](01-provenance-graphs.ipynb) -- Visualize the pipeline you profiled
- [Execution Flow](../../concepts/execution-flow.md) -- Understand the phases that timing analysis measures
- [Configuring Execution](../../how-to-guides/configuring-execution.md) -- Resource and scheduling configuration